## Financial Exclusion Classification Model
##### Objective

We aim to predict whether an individual is financially excluded (yes/no) using classification models:

* Logistic Regression
* Random Forest
* Gradient Boosting

We will:

* Build ML pipelines
* Train models
* Compare performance
* Evaluate using Accuracy, Precision, Recall, ROC-AUC
* Analyze confusion matrix (FP, FN)
* Recommend best model

In [12]:
#Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
import pandas as pd
from pathlib import Path


In [ ]:
# Dataset Overview
# We preview the first rows of the dataset to understand its structure, features, and target variable.
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/finaccess_2021_modeling_base.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,County,location_type,gender,education_level,marital_status,adults,financially_excluded,age,is_youth,is_rural_youth
0,Trans Nzoia,Rural,Female,Completed technical training after secondary s...,Widowed,1 adult Household,0,59,0,0
1,Busia,Rural,Female,Completed technical training after secondary s...,Married/Living with partner,>1 adult Household,0,43,0,0
2,Machakos,Rural,Male,Some primary,Divorced/separated,1 adult Household,1,72,0,0
3,Kisumu,Rural,Male,Primary completed,Single/Never Married,>1 adult Household,0,22,1,1
4,Nyeri,Urban,Male,Primary completed,Married/Living with partner,>1 adult Household,0,36,0,0


In [17]:
list(df.columns)

['County',
 'location_type',
 'gender',
 'education_level',
 'marital_status',
 'adults',
 'financially_excluded',
 'age',
 'is_youth',
 'is_rural_youth']

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20902 entries, 0 to 20901
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   County                20902 non-null  object
 1   location_type         20902 non-null  object
 2   gender                20902 non-null  object
 3   education_level       20902 non-null  object
 4   marital_status        20902 non-null  object
 5   adults                20902 non-null  object
 6   financially_excluded  20902 non-null  int64 
 7   age                   20902 non-null  int64 
 8   is_youth              20902 non-null  int64 
 9   is_rural_youth        20902 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 1.6+ MB


# Splitting Features and Target Variable

We separate the dataset into:
- X: input features
- y: target variable (financial exclusion)

In [20]:
X = df.drop("financially_excluded", axis=1)
y = df["financially_excluded"]

# Identifying Feature Types

We separate columns into:
- Categorical features (need encoding)
- Numerical features (can be scaled if needed)

In [21]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

cat_cols, num_cols

(Index(['County', 'location_type', 'gender', 'education_level',
        'marital_status', 'adults'],
       dtype='object'),
 Index(['age', 'is_youth', 'is_rural_youth'], dtype='object'))

# Preprocessing Pipeline

# Data Preprocessing with ColumnTransformer

We use ColumnTransformer to apply:
- OneHotEncoding for categorical variables
- Scaling for numerical variables

This ensures all models receive properly formatted data.

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

# Model Building

We define three classification models:

1. Logistic Regression → baseline interpretable model
2. Random Forest → ensemble tree-based model
3. Gradient Boosting → advanced boosting model

Each model is wrapped in a pipeline with preprocessing.

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Logistic Regression Pipeline

In [24]:
log_reg = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Random Forest Pipeline

In [25]:
rf = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Gradient Boosting Pipeline

In [26]:
gb = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

# Train-Test Split

We split the dataset into training and testing sets.

We use stratification to preserve class distribution of financial exclusion.

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model Training

We train all models using the training dataset.

Each model learns patterns that predict financial exclusion.

In [28]:
models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

for name, model in models.items():
    model.fit(X_train, y_train)

# Model Evaluation

We evaluate models using:
- Accuracy
- Precision
- Recall
- ROC-AUC

We also visualize:
- Confusion Matrix
- ROC Curve

In [29]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns